In [2]:
# 1. Install necessary tools
!pip install -q kaggle fastai gradio

# 2. Set your Kaggle Token (Copy-paste the token from your screenshot here)
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"
token = {"username":"muaviatanveer","key":"4a8f4144fe954a26066380e63b97c9b2"} # This is derived from your KGAT

import json
with open('/content/kaggle.json', 'w') as f:
    json.dump(token, f)

!chmod 600 /content/kaggle.json

# 3. Download the datasets directly to Colab
print("📦 Downloading datasets... this will be fast on your paid account.")
# Download Competition Data (Check your sidebar name, usually it is this:)
!kaggle competitions download -c sign-language-contest
# Download External 223k Dataset
!kaggle datasets download -d debashishsau/aslamerican-sign-language-aplhabet-dataset

# 4. Unzip
print("📂 Extracting images...")
!unzip -q sign-language-contest.zip -d competition_data
!unzip -q aslamerican-sign-language-aplhabet-dataset.zip -d external_data
print("✅ DATA IS READY!")

📦 Downloading datasets... this will be fast on your paid account.
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
Dataset URL: https://www.kaggle.com/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset
License(s): CC0-1.0
100% 4.20G/4.20G [01:53<00:00, 39.6MB/s]

📂 Extracting images...
unzip:  cannot find or open sign-language-contest.zip, sign-language-contest.zip.zip or sign-language-contest.zip.ZIP.
✅ DATA IS READY!


In [3]:
from fastai.vision.all import *
import os

# 1. 🔍 SMART SEARCH FOR ALL IMAGES
# This will find images in BOTH competition_data and external_data automatically
print("🔍 Searching for all training images in Colab...")
train_files = []
for root, dirs, files in os.walk('/content'):
    # Avoid test folders and sample_data
    if ('train' in root.lower()) and ('sample_data' not in root.lower()):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                train_files.append(Path(os.path.join(root, f)))

print(f"📊 TOTAL IMAGES SECURED FOR TRAINING: {len(train_files)}")

if len(train_files) == 0:
    print("🚨 ERROR: No images found. Check your folder sidebar on the left!")
    exit()

# 2. LABEL FIXER
def get_label(file_path):
    name = Path(file_path).parent.name.lower()
    if 'space' in name: return 'space'
    if 'nothing' in name: return 'nothing'
    if 'del' in name or 'delete' in name: return 'del'
    return name[-1].upper() if len(name) > 1 and name[-1].isalpha() else name.upper()

# 3. LOAD DATA (High Resolution for the Demo)
print("📦 Loading Data into Brain...")
dls = ImageDataLoaders.from_path_func(
    path=".", fnames=train_files, label_func=get_label,
    valid_pct=0.05, seed=42, item_tfms=Resize(224),
    batch_tfms=aug_transforms(do_flip=False, max_lighting=0.3),
    bs=128
)

# 4. TRAIN (Colab Pro will make this fast)
print("🧠 Training the Nuclear Brain... Ready for the presentation.")
learn = vision_learner(dls, resnet50, metrics=[accuracy]).to_fp16()
learn.fine_tune(3)

# 5. EXPORT
learn.export('nuclear_brain.pkl')
print("✅ SUCCESS! nuclear_brain.pkl is ready for your demo.")

🔍 Searching for all training images in Colab...
📊 TOTAL IMAGES SECURED FOR TRAINING: 223074
📦 Loading Data into Brain...
🧠 Training the Nuclear Brain... Ready for the presentation.
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 237MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,0.295479,0.136331,0.958755,03:11


epoch,train_loss,valid_loss,accuracy,time
0,0.025904,0.012918,0.996234,03:42
1,0.009209,0.004309,0.999193,03:42
2,0.002371,0.003264,0.999283,03:42


✅ SUCCESS! nuclear_brain.pkl is ready for your demo.


In [9]:
import gradio as gr
from fastai.vision.all import *
import json
import os

# --- DATABASE LOGIC ---
USER_DB_FILE = "users_db.json"

def load_users():
    if not os.path.exists(USER_DB_FILE):
        return {"admin": "bridge2024"} # Default user
    with open(USER_DB_FILE, "r") as f:
        return json.load(f)

def save_user(username, password):
    users = load_users()
    if username in users:
        return False, "Username already exists!"
    users[username] = password
    with open(USER_DB_FILE, "w") as f:
        json.dump(users, f)
    return True, "Account created successfully! Please Login."

def authenticate(username, password):
    users = load_users()
    if username in users and users[username] == password:
        return True
    return False

# --- AI LOGIC ---
learn = load_learner('nuclear_brain.pkl')

def predict_and_spell(img, current_text, buffer):
    if img is None: return "", current_text, buffer

    pred, _, _ = learn.predict(img)
    label = str(pred)

    buffer.append(label)
    if len(buffer) > 3: buffer.pop(0)

    if buffer.count(label) == 3:
        if label == 'space':
            if not current_text.endswith(" "): current_text += " "
        elif label == 'del':
            current_text = current_text[:-1]
        elif label == 'nothing': pass
        else:
            if not current_text.endswith(label): current_text += label
        buffer = []

    return label, current_text, buffer

# --- STYLING ---
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700&family=Inter:wght@300;600&display=swap');

body, .gradio-container {
    background: linear-gradient(135deg, #090a0f, #1b2735) !important;
    font-family: 'Inter', sans-serif !important;
    color: #ffffff !important;
}

.glass-panel {
    background: rgba(255, 255, 255, 0.03) !important;
    backdrop-filter: blur(15px) !important;
    border-radius: 20px !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.8) !important;
    padding: 40px !important;
    margin: 20px auto !important;
}

.auth-card {
    max-width: 450px !important;
    margin: 100px auto !important;
}

.title-text {
    font-family: 'Orbitron', sans-serif !important;
    text-align: center;
    color: #ffffff;
    text-shadow: 0 0 15px #0984E3;
}

.detected-label textarea {
    font-family: 'Orbitron', sans-serif !important;
    font-size: 70px !important;
    text-align: center !important;
    color: #00f2fe !important;
    background: transparent !important;
    border: none !important;
}

.message-box textarea {
    font-size: 32px !important;
    color: #00e676 !important;
    background: rgba(0, 0, 0, 0.4) !important;
    border: 1px solid #00e676 !important;
}

.primary-btn {
    background: linear-gradient(45deg, #0984E3, #00f2fe) !important;
    border: none !important;
    color: white !important;
    font-weight: bold !important;
}
"""

# --- UI BUILD ---
with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:

    # Session States
    text_state = gr.State("")
    buffer_state = gr.State([])

    # 1. LOGIN / REGISTER UI
    with gr.Column(visible=True, elem_classes="glass-panel auth-card") as auth_panel:
        gr.Markdown("# 🌉 SARA'S BRIDGE\n### Secure Neural Portal", elem_classes="title-text")

        with gr.Tabs():
            with gr.TabItem("Login"):
                login_user = gr.Textbox(label="Username", placeholder="Enter username...")
                login_pw = gr.Textbox(label="Password", type="password")
                login_btn = gr.Button("UNLOCk ACCESS", elem_classes="primary-btn")
                login_msg = gr.Markdown()

            with gr.TabItem("Register"):
                reg_user = gr.Textbox(label="New Username", placeholder="Choose a name...")
                reg_pw = gr.Textbox(label="New Password", type="password")
                reg_btn = gr.Button("CREATE ACCOUNT", variant="secondary")
                reg_msg = gr.Markdown()

    # 2. MAIN APPLICATION UI (Hidden by default)
    with gr.Column(visible=False, elem_classes="glass-panel") as main_app:
        gr.Markdown("# 🌉 Sara's Bridge", elem_classes="title-text")
        gr.Markdown("<p style='text-align:center'>AI-Powered Sign Language Translation Active</p>")

        with gr.Row():
            with gr.Column(scale=2):
                input_img = gr.Image(sources=["webcam"], streaming=True, label="Neural Feed")

            with gr.Column(scale=1):
                gr.Markdown("### 🧠 Interpretation")
                out_label = gr.Textbox(label="", elem_classes="detected-label")
                clear_btn = gr.Button("🗑️ PURGE", variant="stop")

        gr.Markdown("### 📡 Spelled Message")
        out_text = gr.Textbox(label="", placeholder="System ready...", elem_classes="message-box")

    # --- BUTTON LOGIC ---

    # Register logic
    def handle_register(u, p):
        if len(u) < 3 or len(p) < 3:
            return "Username/Password too short!"
        success, msg = save_user(u, p)
        return msg

    reg_btn.click(handle_register, [reg_user, reg_pw], reg_msg)

    # Login logic
    def handle_login(u, p):
        if authenticate(u, p):
            # Hide auth panel, show main app
            return gr.update(visible=False), gr.update(visible=True), f"Welcome, {u}!"
        return gr.update(visible=True), gr.update(visible=False), "Invalid Credentials"

    login_btn.click(
        handle_login,
        [login_user, login_pw],
        [auth_panel, main_app, login_msg]
    )

    # Core Logic
    input_img.stream(
        fn=predict_and_spell,
        inputs=[input_img, out_text, buffer_state],
        outputs=[out_label, out_text, buffer_state]
    )

    clear_btn.click(
        lambda: ("", "", []),
        outputs=[out_label, out_text, buffer_state]
    )

# Launch
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")
/tmp/ipykernel_2793/2468361153.py:111: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:
/tmp/ipykernel_2793/2468361153.py:111: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() in

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://991a0432c33eb6fd73.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
